In [ ]:
!git clone -q --depth 1 https://github.com/tangsanli5201/DeepPCB.git
%cd DeepPCB
!ls

In [ ]:
%%writefile pcb_common.py
"""Shared preprocessing for training AND inference (one code path, no drift).

Pipeline for a photo: localize board -> crop + rescale to DeepPCB scale (640 px wide)
-> contour candidates -> prep() each candidate -> CNN (7 classes, last = background).
"""
import cv2
import numpy as np

CLASS_NAMES = ["copper", "mousebite", "open", "pin-hole", "short", "spur", "background"]
BG = 6
LABEL_TO_IDX = {1: 2, 2: 4, 3: 1, 4: 5, 5: 0, 6: 3}  # DeepPCB label id -> CLASS_NAMES index
BOARD_W = 640  # DeepPCB image width; candidate size limits below assume this scale


def prep(gray_crop):
    """128x128 float32, contrast-stretched per crop so lighting/colour polarity gaps shrink."""
    g = cv2.resize(gray_crop, (128, 128), interpolation=cv2.INTER_AREA).astype(np.float32)
    lo, hi = np.percentile(g, (2, 98))
    g = np.clip((g - lo) / max(hi - lo, 8.0), 0, 1) * 255.0
    return g[..., None]


def candidates(gray, mask=None):
    """Same contour candidate logic as app.py, optionally restricted to the board mask."""
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    th = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    if mask is not None:
        th = cv2.bitwise_and(th, mask)
    cs, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cs:
        x, y, w, h = cv2.boundingRect(c)
        if not (20 < w < 300 and 20 < h < 300):
            continue
        per = cv2.arcLength(c, True)
        circ = 4 * np.pi * cv2.contourArea(c) / (per * per) if per > 0 else 0.0
        if circ > 0.75:
            continue
        out.append((x, y, w, h))
    return out


def windows(gray, mask=None, sizes=(48, 72), stride=24):
    """Sliding square windows over the board (replaces contour candidates, which missed ~60% of defects)."""
    H, W = gray.shape
    out = []
    for s in sizes:
        for y in range(0, H - s + 1, stride):
            for x in range(0, W - s + 1, stride):
                if mask is not None and mask[y + s // 2, x + s // 2] == 0:
                    continue
                out.append((x, y, s, s))
    return out


def nms(dets, thr=0.3):
    """dets: (x,y,w,h,name,conf). Greedy by confidence."""
    keep = []
    for d in sorted(dets, key=lambda t: -t[5]):
        ok = True
        for k in keep:
            ix = max(0, min(d[0] + d[2], k[0] + k[2]) - max(d[0], k[0]))
            iy = max(0, min(d[1] + d[3], k[1] + k[3]) - max(d[1], k[1]))
            if ix * iy / (d[2] * d[3] + k[2] * k[3] - ix * iy) > thr:
                ok = False; break
        if ok:
            keep.append(d)
    return keep


def localize_board(bgr):
    """Return (x, y, w, h, mask) of the board in the photo, or None if no board found.

    Colour boards: the solder-mask hue is the saturation-weighted hue peak.
    Dark/grey boards: Otsu on grey. Largest blob wins, holes filled.
    ponytail: classical heuristic; swap for a trained board segmenter if it fails on your photos.
    """
    h, w = bgr.shape[:2]
    s = 640.0 / max(h, w)
    small = cv2.resize(bgr, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(cv2.GaussianBlur(small, (5, 5), 0), cv2.COLOR_BGR2HSV)
    hue, sat, val = cv2.split(hsv)
    sat_ok = (sat > 60) & (val > 40)
    cand = []
    if sat_ok.mean() > 0.03:
        hist = np.bincount(hue[sat_ok].ravel(), minlength=180).astype(np.float32)
        hist = np.convolve(np.r_[hist[-8:], hist, hist[:8]], np.ones(17), "same")[8:-8]
        peak = int(np.argmax(hist))
        d = np.minimum(np.abs(hue.astype(int) - peak), 180 - np.abs(hue.astype(int) - peak))
        cand.append(((d < 14) & sat_ok).astype(np.uint8) * 255)
    g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    _, otsu = cv2.threshold(cv2.GaussianBlur(g, (7, 7), 0), 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cand.append(otsu)
    best = None
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    for m in cand:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=2)
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cs:
            continue
        c = max(cs, key=cv2.contourArea)
        frac = cv2.contourArea(c) / m.size
        if 0.08 < frac < 0.97 and (best is None or frac > best[0]):
            best = (frac, c)
    if best is None:
        return None
    mask = np.zeros(g.shape, np.uint8)
    cv2.drawContours(mask, [cv2.convexHull(best[1])], -1, 255, -1)
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    x, y, bw, bh = cv2.boundingRect(mask)
    return x, y, bw, bh, mask


def detect(model, bgr, conf=0.80, use_localization=True):
    """Full pipeline. Returns list of (x, y, w, h, class_name, confidence) in ORIGINAL image coords."""
    H, W = bgr.shape[:2]
    mask_full = None
    ox = oy = 0
    img = bgr
    if use_localization:
        loc = localize_board(bgr)
        if loc is None:
            return []  # no board in frame -> nothing to report
        x, y, bw, bh, mask_full = loc
        img, ox, oy = bgr[y:y + bh, x:x + bw], x, y
        mask_full = mask_full[y:y + bh, x:x + bw]
    sc = BOARD_W / img.shape[1] if use_localization else 1.0
    img_s = cv2.resize(img, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img_s, cv2.COLOR_BGR2GRAY)
    m = None
    if mask_full is not None:
        m = cv2.erode(cv2.resize(mask_full, (gray.shape[1], gray.shape[0]), interpolation=cv2.INTER_NEAREST),
                      np.ones((9, 9), np.uint8))
    boxes = windows(gray, m)
    if not boxes:
        return []
    batch = np.stack([prep(gray[y:y + h, x:x + w]) for x, y, w, h in boxes])
    probs = model.predict(batch, verbose=0)
    out = []
    for (x, y, w, h), p in zip(boxes, probs):
        i = int(np.argmax(p))
        if i == BG or p[i] < conf:
            continue
        out.append((int(x / sc) + ox, int(y / sc) + oy, int(w / sc), int(h / sc), CLASS_NAMES[i], float(p[i])))
    return nms(out)


In [ ]:
%%writefile 6_train_robust.py
"""Train the 7-class (6 defects + background) CNN with a BOARD-LEVEL split.

Validation boards come from whole held-out DeepPCB groups, so no template is shared with training.
Output: pcb_cnn_model_v2.keras (original pcb_cnn_model.keras is untouched).
Run from inside this folder:  python 6_train_robust.py
"""
import glob, os, random
import cv2, numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from pcb_common import candidates, CLASS_NAMES, BG, LABEL_TO_IDX, prep

VAL_GROUPS = ["group50600", "group90100"]   # held out entirely
random.seed(0); np.random.seed(0); tf.random.set_seed(0)
R = lambda im: cv2.resize(im, (128, 128), interpolation=cv2.INTER_AREA)


def read_boxes(p):
    out = []
    for line in open(p):
        v = line.split()
        if len(v) == 5:
            out.append(tuple(map(int, v)))
    return out


def synth_bg(n):
    """Non-board junk: flat, gradient, noisy, blobby, stripe patches (table, skin, shadows, text-ish)."""
    out = []
    for _ in range(n):
        base = np.random.randint(0, 256)
        im = np.full((128, 128), base, np.float32)
        k = random.choice(["flat", "grad", "noise", "blob", "stripe"])
        if k == "grad":
            im += np.linspace(0, np.random.randint(-120, 120), 128)[None, :] * np.random.choice([0, 1])
            im += np.linspace(0, np.random.randint(-120, 120), 128)[:, None]
        elif k == "noise":
            im += np.random.randn(128, 128) * np.random.randint(5, 40)
        elif k == "blob":
            for _ in range(random.randint(1, 4)):
                cv2.circle(im, tuple(np.random.randint(0, 128, 2).tolist()), random.randint(8, 60),
                           float(np.random.randint(0, 256)), -1)
            im = cv2.GaussianBlur(im, (0, 0), random.uniform(0, 6))
        elif k == "stripe":
            for _ in range(random.randint(1, 6)):
                cv2.line(im, tuple(np.random.randint(0, 128, 2).tolist()), tuple(np.random.randint(0, 128, 2).tolist()),
                         float(np.random.randint(0, 256)), random.randint(1, 12))
        out.append(np.clip(im, 0, 255).astype(np.uint8))
    return out


def win(img, cx, cy, side):
    h, w = img.shape; s = int(max(24, side)); x = int(np.clip(cx - s / 2, 0, max(0, w - s))); y = int(np.clip(cy - s / 2, 0, max(0, h - s)))
    return R(img[y:y + s, x:x + s])


def build(groups):
    """Window-style crops (same geometry as inference sliding windows)."""
    X, y = [], []
    for g in groups:
        for tp in glob.glob(f"PCBData/{g}/*/*_test.jpg"):
            bid = os.path.basename(tp)[:-9]
            lp = glob.glob(f"PCBData/{g}/*/{bid}.txt")
            test = cv2.imread(tp, 0); tmp = cv2.imread(tp.replace("_test", "_temp"), 0)
            if test is None or tmp is None or not lp:
                continue
            boxes = [b for b in read_boxes(lp[0]) if b[4] in LABEL_TO_IDX]
            H, W = test.shape
            for x1, y1, x2, y2, t in boxes:
                for _ in range(3):
                    side = max(x2 - x1, y2 - y1) * random.uniform(1.4, 2.4)
                    cx = (x1 + x2) / 2 + random.uniform(-.2, .2) * side; cy = (y1 + y2) / 2 + random.uniform(-.2, .2) * side
                    X.append(win(test, cx, cy, max(side, 40))); y.append(LABEL_TO_IDX[t])
            for _ in range(8):   # defect-free template windows
                side = random.uniform(40, 80); X.append(win(tmp, random.uniform(0, W), random.uniform(0, H), side)); y.append(BG)
            n = 0
            while n < 6:         # test-image windows far from every defect
                side = random.uniform(40, 80); cx, cy = random.uniform(0, W), random.uniform(0, H)
                if any(a - .6 * side < cx < c + .6 * side and b - .6 * side < cy < d + .6 * side for a, b, c, d, _ in boxes):
                    if random.random() < .9: continue
                X.append(win(test, cx, cy, side)); y.append(BG); n += 1
    return X, y


def augment(im):
    im = im.astype(np.float32)
    if random.random() < .5: im = 255 - im                       # polarity (colour boards map differently to grey)
    if random.random() < .5: im = np.fliplr(im)
    if random.random() < .5: im = np.flipud(im)
    im = np.rot90(im, random.randint(0, 3))
    a = random.uniform(.5, 1.5); b = random.uniform(-60, 60)    # contrast / brightness
    im = im * a + b
    im = 255 * (np.clip(im, 0, 255) / 255) ** random.uniform(.6, 1.6)   # gamma
    if random.random() < .5: im = cv2.GaussianBlur(im, (0, 0), random.uniform(.3, 2.2))
    if random.random() < .5: im = im + np.random.randn(*im.shape) * random.uniform(2, 14)
    if random.random() < .4:                                     # resolution loss
        s = random.randint(24, 80); im = cv2.resize(cv2.resize(im, (s, s)), (128, 128))
    M = cv2.getRotationMatrix2D((64, 64), random.uniform(-15, 15), random.uniform(.85, 1.15))
    im = cv2.warpAffine(im, M, (128, 128), borderMode=cv2.BORDER_REFLECT)
    return np.clip(im, 0, 255).astype(np.uint8)


class Seq(tf.keras.utils.Sequence):
    def __init__(self, X, y, train, bs=64):
        super().__init__(); self.X, self.y, self.train, self.bs = X, np.array(y), train, bs
        self.idx = np.arange(len(X)); self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.X) / self.bs))
    def on_epoch_end(self):
        if self.train: np.random.shuffle(self.idx)
    def __getitem__(self, i):
        ids = self.idx[i * self.bs:(i + 1) * self.bs]
        xs = [augment(self.X[j]) if self.train else self.X[j] for j in ids]
        return np.stack([prep(x) for x in xs]), self.y[ids]


if __name__ == "__main__":
    all_groups = sorted(os.path.basename(p) for p in glob.glob("PCBData/group*"))
    tr_groups = [g for g in all_groups if g not in VAL_GROUPS]
    Xtr, ytr = build(tr_groups); Xva, yva = build(VAL_GROUPS)
    s = synth_bg(1500); Xtr += s; ytr += [BG] * len(s)
    s = synth_bg(300); Xva += s; yva += [BG] * len(s)
    print("train groups:", tr_groups, "val groups:", VAL_GROUPS)
    print("train counts:", dict(zip(CLASS_NAMES, np.bincount(ytr, minlength=7))))
    print("val counts:  ", dict(zip(CLASS_NAMES, np.bincount(yva, minlength=7))))
    # background is the most diverse class -> weight it up a little
    cw = {i: 1.0 for i in range(7)}; cw[BG] = 1.5
    model = models.Sequential([
        layers.Rescaling(1. / 255, input_shape=(128, 128, 1)),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dropout(0.5),
        layers.Dense(7, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(Seq(Xtr, ytr, True), validation_data=Seq(Xva, yva, False), epochs=12, class_weight=cw,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    model.save("pcb_cnn_model_v2.keras")
    print("saved pcb_cnn_model_v2.keras")


In [ ]:
!python 6_train_robust.py 2>&1 | grep -E '^(train|val)|val_accuracy|saved'

In [ ]:
from google.colab import files
files.download('pcb_cnn_model_v2.keras')